### In your VS Code Jupyter notebook

In [ ]:
import sys

print(sys.executable)
print(sys.version)

In [ ]:
import pyspark

print(pyspark.__version__)

#### Start a Spark session connected to MinIO

In [ ]:
from pyspark.sql import SparkSession

MINIO_ENDPOINT = "http://localhost:9000"
MINIO_ACCESS_KEY = "minioadmin"
MINIO_SECRET_KEY = "minioadmin"

spark = (
    SparkSession.builder
    .appName("RestaurantDataCheck")
    .config(
        "spark.hadoop.fs.s3a.endpoint",
        MINIO_ENDPOINT
    )
    .config(
        "spark.hadoop.fs.s3a.access.key",
        MINIO_ACCESS_KEY
    )
    .config(
        "spark.hadoop.fs.s3a.secret.key",
        MINIO_SECRET_KEY
    )
    .config(
        "spark.hadoop.fs.s3a.path.style.access",
        "true"
    )
    .config(
        "spark.hadoop.fs.s3a.connection.ssl.enabled",
        "false"
    )
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

#### Check the generated directory

In [ ]:
spark.read.format("binaryFile").load(
    "s3a://rawload/Restaurant_Food_Delivery_Test/categories.csv/"
).select("path", "length").show(truncate=False)

#### read the generated data

In [ ]:
df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(
        "s3a://rawload/Restaurant_Food_Delivery_Test/categories.csv/"
    )
)

In [ ]:
df.printSchema()

#### Check the actual dates

In [ ]:
from pyspark.sql import functions as F

df.select(
    "id",
    "created_at",
    "updated_at"
).show(20, truncate=False)

##### Source

In [ ]:
source_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(
        "s3a://rawload/Restaurant_Food_Delivery/categories.csv"
    )
)

##### Transformed

In [ ]:
test_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(
        "s3a://rawload/Restaurant_Food_Delivery_Test/categories.csv/"
    )
)

##### Output

In [ ]:
source_df.select(
    "id",
    "created_at",
    "updated_at"
).show(10, truncate=False)

test_df.select(
    "id",
    "created_at",
    "updated_at"
).show(10, truncate=False)

#### Better: let Spark prove the difference

##### join the source and transformed versions:

In [ ]:
comparison = (
    source_df.alias("src")
    .join(
        test_df.alias("tgt"),
        F.col("src.id") == F.col("tgt.id"),
        "inner"
    )
    .select(
        F.col("src.id"),
        F.col("src.created_at").alias("source_created_at"),
        F.col("tgt.created_at").alias("test_created_at"),
        F.col("src.updated_at").alias("source_updated_at"),
        F.col("tgt.updated_at").alias("test_updated_at")
    )
)

comparison.show(20, truncate=False)

##### calculate the difference

In [ ]:
comparison.select(
    "id",
    "source_created_at",
    "test_created_at",
    F.months_between(
        "test_created_at",
        "source_created_at"
    ).alias("months_difference")
).show(20, truncate=False)

#### Most important test: `order_status_history`

In [ ]:
osh = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(
        "s3a://rawload/Restaurant_Food_Delivery/order_status_history.csv"
    )
)

In [ ]:
osh.printSchema()

In [ ]:
osh.select(
    F.min("created_at").alias("min_created_at"),
    F.max("created_at").alias("max_created_at"),
    F.count("*").alias("record_count")
).show()

In [ ]:
osh_source = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(
        "s3a://rawload/Restaurant_Food_Delivery/order_status_history.csv"
    )
)

osh_source.select(
    F.min("created_at").alias("min_created_at"),
    F.max("created_at").alias("max_created_at"),
    F.count("*").alias("record_count")
).show()

In [ ]:
osh_updated = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(
        "s3a://rawload/Restaurant_Food_Delivery_Test/order_status_history.csv"
    )
)

osh_updated.select(
    F.min("created_at").alias("min_created_at"),
    F.max("created_at").alias("max_created_at"),
    F.count("*").alias("record_count")
).show()